# Melanoma experiments: does the external ISIC 2019 data help?

This notebook runs the baseline sweep on Kaggle. It trains a ResNet34 twice over
five patient-grouped folds -- once on the competition data alone, once with ISIC
2019 added to the training side -- and reports the difference.

Ten training runs, about three GPU-hours in total.

**Settings on the right: Accelerator = GPU T4 x2, Internet = On.**

Not P100. Kaggle still offers it, but current PyTorch builds no longer compile
for Pascal: `torch.cuda.is_available()` returns `True` and then the first real
kernel launch fails. Cell 1 checks for this.

Everything is resumable. Each finished fold is appended to `results.csv` as it
completes, so if the session dies you can open it again and Run All, and it will
skip whatever already finished.

The written-up findings are in [reports/Experiment_Report.md](../reports/Experiment_Report.md).

## 1. Check the environment

Two things have to be true before anything else is worth running: a working GPU,
and an albumentations new enough to have `A.Affine` (the 1.x name was
`ShiftScaleRotate`, and the training code uses the 2.x spelling).

In [9]:
import os, shutil, subprocess, time, glob
t0 = time.time()

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU T4 x2, then Run All again."

# is_available() is not enough on Kaggle. A P100 is sm_60 and current PyTorch
# builds no longer compile for it: is_available() returns True and then every
# kernel launch fails. Check the architecture is actually one this build has.
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
assert arch in torch.cuda.get_arch_list(), (
    f"{name} is {arch}, which this PyTorch does not support "
    f"({torch.cuda.get_arch_list()}). Switch the accelerator to GPU T4 x2.")
print("gpu  :", name, f"({arch})")

# The training code uses the albumentations 2.x spelling A.Affine. In 1.x the
# same transform was called ShiftScaleRotate, so an old version imports fine and
# then fails at the first augmented batch.
try:
    import albumentations as A
    A.Affine
    print("albumentations", A.__version__)
except Exception:
    subprocess.run(["pip", "install", "-q", "albumentations>=2.0"], check=False)
    import albumentations as A
    print("albumentations installed:", A.__version__)


torch 2.10.0+cu128 | cuda True
gpu: Tesla T4
albumentations 2.0.8


## 2. Find the data and the code

Kaggle mounts datasets under `/kaggle/input/<slug>/`, but it also auto-extracts
`.tar` uploads, which adds another directory level, so the path is not
predictable. Walking the tree is more robust than assuming a depth.

The pruning line matters. `train_512` holds 57,855 files on a network mount, and
descending into it turns a two-second search into several minutes.

The `.py` files are copied out of the read-only input mount into `/kaggle/working`
so they can be imported and run.

In [10]:
import os, shutil

FOLDS = None
IMG_DIR = None
py_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    if IMG_DIR is None and "train_512" in dirs:
        IMG_DIR = os.path.join(root, "train_512")
    if FOLDS is None and "folds.csv" in files:
        FOLDS = os.path.join(root, "folds.csv")
    py_files += [os.path.join(root, f) for f in files if f.endswith(".py")]
    dirs[:] = [d for d in dirs if d not in ("train_512", "test_512")]

assert FOLDS, "folds.csv not found"
assert IMG_DIR, "train_512 not found"

INPUT = os.path.dirname(FOLDS)
WORK = "/kaggle/working"
SRC = os.path.join(WORK, "src")
os.makedirs(SRC, exist_ok=True)
for p in py_files:
    shutil.copy(p, SRC)

print("input   :", INPUT)
print("img_dir :", IMG_DIR)
print("folds   :", FOLDS)
print("copied  :", len(py_files), "source files")
assert len(py_files) >= 5, "source .py files missing"

input   : /kaggle/input/datasets/harutkesablyan/melanoma-512-preprocessed
img_dir : /kaggle/input/datasets/harutkesablyan/melanoma-512-preprocessed/melanoma_512/melanoma_512/train_512
folds   : /kaggle/input/datasets/harutkesablyan/melanoma-512-preprocessed/folds.csv
copied  : 7 source files


## 3. Shrink the photos to 224 once, up front

The photos on disk are 512x512 but the baseline trains at 224. Resizing inside
the DataLoader would redo the same work every epoch, ten times per run, twenty
times across both experiments.

`IMREAD_REDUCED_COLOR_2` is the trick worth knowing here: it decodes the JPEG
straight to half size using DCT scaling, which is much cheaper than decoding the
full 512x512 and resizing afterwards.

Writing to a `.tmp` file and renaming afterwards means a session that dies
mid-write leaves no half-written JPEG that the next run would treat as finished.

In [16]:
import os, cv2, numpy as np
from multiprocessing import Pool

FAST_DIR = "/kaggle/working/train_224"
os.makedirs(FAST_DIR, exist_ok=True)

names = [f for f in os.listdir(IMG_DIR) if f.endswith(".jpg")]
todo = [n for n in names if not os.path.exists(os.path.join(FAST_DIR, n))]
print(f"{len(names)} photos, {len(todo)} still to convert")

def shrink(name):
    src = os.path.join(IMG_DIR, name)
    dst = os.path.join(FAST_DIR, name)
    # IMREAD_REDUCED_COLOR_2 decodes the JPEG straight to half size using DCT
    # scaling, which is far cheaper than decoding 512x512 and resizing after.
    img = cv2.imread(src, cv2.IMREAD_REDUCED_COLOR_2)
    if img is None:
        return name
    img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
    tmp = dst + ".tmp"
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    if not ok:
        return name
    with open(tmp, "wb") as h:
        h.write(buf.tobytes())
    os.rename(tmp, dst)
    return None

if todo:
    with Pool(4) as pool:
        bad = [r for r in pool.imap_unordered(shrink, todo, chunksize=64) if r]
    print("failed:", len(bad))

print("ready:", len(os.listdir(FAST_DIR)), "photos in", FAST_DIR)
assert len(os.listdir(FAST_DIR)) == len(names)

57855 photos, 57855 still to convert
failed: 0
ready: 57855 photos in /kaggle/working/train_224


## 4. Run the sweep

`experiment_runner.py` does the actual work: ten runs, two experiments across
five folds each, writing one row to `results.csv` per finished fold.

`--time_budget_h` makes it stop cleanly rather than being killed mid-fold when
Kaggle's session limit arrives. Rerunning picks up where it stopped.

The only difference between the two experiments is whether the ISIC 2019 rows
are in the training split. Same folds, same seed, same augmentation, same
schedule -- which is what makes the comparison a paired one, and why the
fold-to-fold difficulty cancels out of the result.

In [17]:
BUDGET = 5.0

!cd {SRC} && python experiment_runner.py \
    --img_dir "{FAST_DIR}" \
    --folds_csv "{FOLDS}" \
    --out_dir {WORK}/reports \
    --only resnet34_224_ext,resnet34_224_noext \
    --folds 0,1,2,3,4 \
    --batch_size 96 \
    --num_workers 4 \
    --time_budget_h {BUDGET}

device: cuda
gpu: Tesla T4
  image check: 200 sampled photos all present in /kaggle/working/train_224

=== resnet34_224_ext | fold 0 | 0.00h elapsed ===
    epoch  1/10  loss 0.8165  PR-AUC 0.1154  ROC-AUC 0.7884
    epoch  2/10  loss 0.7317  PR-AUC 0.1293  ROC-AUC 0.8491
    epoch  3/10  loss 0.6849  PR-AUC 0.1388  ROC-AUC 0.8736
    epoch  4/10  loss 0.6508  PR-AUC 0.1704  ROC-AUC 0.8758
    epoch  5/10  loss 0.6037  PR-AUC 0.2030  ROC-AUC 0.8770
    epoch  6/10  loss 0.5661  PR-AUC 0.2081  ROC-AUC 0.8828
    epoch  7/10  loss 0.5294  PR-AUC 0.1972  ROC-AUC 0.8854
    epoch  8/10  loss 0.4867  PR-AUC 0.2078  ROC-AUC 0.8868
    epoch  9/10  loss 0.4615  PR-AUC 0.2115  ROC-AUC 0.8903
    epoch 10/10  loss 0.4486  PR-AUC 0.2123  ROC-AUC 0.8846
  -> PR-AUC 0.2123 | ROC-AUC 0.8846 | 22.3 min

=== resnet34_224_ext | fold 1 | 0.37h elapsed ===
    epoch  1/10  loss 0.8239  PR-AUC 0.0936  ROC-AUC 0.8204
    epoch  2/10  loss 0.7313  PR-AUC 0.1171  ROC-AUC 0.8612
    epoch  3/10  loss 0.6850 

## 5. Report

`report_results.py` turns `results.csv` into the per-fold and averaged tables,
and runs the paired significance tests.

The headline: with ISIC 2019 the model scores PR-AUC 0.2285 against 0.1873
without it, and it wins on all five folds. At a fixed 5% false-alarm rate that is
303 melanomas found instead of 273.

In [18]:
# --- report --------------------------------------------------------------
!cd {SRC} && python report_results.py --in_dir {WORK}/reports

import pandas as pd
pd.read_csv(f"{WORK}/reports/results.csv")


# Experiment results

10 runs across 2 experiments, 2.9 GPU-hours total.

## Summary, averaged over folds

| experiment | folds | PR-AUC | ROC-AUC | Sens@95Spec | minutes |
| --- | ---: | ---: | ---: | ---: | ---: |
| `resnet34_224_ext` | 5 | 0.2285 ± 0.0477 | 0.8873 | 0.5189 | 112 |
| `resnet34_224_noext` | 5 | 0.1873 ± 0.0283 | 0.8809 | 0.4674 | 63 |

A random model scores about **0.0176** PR-AUC here, so that is the number to beat.

## Did ISIC 2019 actually help?

Same folds, same seed, one variable changed.

| fold | with external | without | difference |
| ---: | ---: | ---: | ---: |
| 0 | 0.2123 | 0.1632 | +0.0492 |
| 1 | 0.1589 | 0.1570 | +0.0019 |
| 2 | 0.2863 | 0.1980 | +0.0882 |
| 3 | 0.2539 | 0.2271 | +0.0268 |
| 4 | 0.2311 | 0.1912 | +0.0400 |
| **mean** | | | **+0.0412** |

External data wins on **5 of 5 folds**.

Consistent across every fold, so this is a real effect. Keep the external data.

## Out-of-fold scores

Every competition photo scored exactly once, by the mode

,experiment,model,image_size,external,fold,epochs,best_epoch,pr_auc,roc_auc,sens_at_95_spec,best_threshold,f1_at_best_threshold,n_train,n_val,val_positives,pos_weight,minutes
0,resnet34_224_ext,resnet34,224,True,0,10,10,0.212330,0.884590,0.504274,0.705032,0.333333,51228,6627,117,9.410079,22.313875
1,resnet34_224_ext,resnet34,224,True,1,10,10,0.158904,0.870441,0.465517,0.706642,0.281481,51230,6625,116,9.408371,22.385406
2,resnet34_224_ext,resnet34,224,True,2,10,7,0.286251,0.903949,0.586207,0.682139,0.365957,51234,6621,116,9.409184,22.354017
3,resnet34_224_ext,resnet34,224,True,3,10,7,0.253924,0.897193,0.529915,0.627586,0.309038,51227,6628,117,9.409876,22.233182
4,resnet34_224_ext,resnet34,224,True,4,10,5,0.231149,0.880489,0.508475,0.642249,0.244131,51230,6625,118,9.412601,22.370035
5,resnet34_224_noext,resnet34,224,False,0,10,8,0.163171,0.862773,0.435897,0.924605,0.237410,26499,6627,117,55.743042,12.489871
6,resnet34_224_noext,resnet34,224,False,1,10,10,0.157043,0.879942,0.439655,0.871118,0.245232,26501,6625,116,55.626068,12.477798
7,resnet34_224_noext,resnet34,224,False,2,10,9,0.198046,0.885483,0.465517,0.912930,0.289963,26505,6621,116,55.634617,12.531216
8,resnet34_224_noext,resnet34,224,False,3,10,7,0.227105,0.896356,0.512821,0.852810,0.274611,26498,6628,117,55.740898,12.560111
9,resnet34_224_noext,resnet34,224,False,4,10,10,0.191155,0.879736,0.483051,0.934026,0.255924,26501,6625,118,55.869099,12.532213


## 6. Clean up

Kaggle caps the output directory at 20 GB, and the 57,855 shrunk photos are most
of that. Deleting them keeps the saved version of this notebook small enough to
commit. `results.csv` and the report are what we actually want to keep.

In [ ]:
import shutil, os

shutil.rmtree("/kaggle/working/train_224", ignore_errors=True)
shutil.rmtree("/kaggle/working/src", ignore_errors=True)

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1024:10.1f} KB  {p}")